# Aplicações do BlindSet e dos modelos treinados.
- Este notebook contém 2 exercícios de aplicação do Blind_set e dos modelos treinados.
    - 1 cenário onde necessita-se, principalmente, de alto recall da classe positiva (non-fiction).
    - 1 cenário onde necessita-se, principalmente, de alta precisão da classe positiva (non-fiction)
- Com resultados bem satisfatórios

In [1]:
import pandas as pd
import numpy as np
import os, json
from pathlib import Path
import joblib
import  warnings
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*", category=UserWarning)
pd.set_option('future.no_silent_downcasting', True)

# Blind Set
## Load and describe

In [2]:
# Read blind set (sem genero rotulado)
path_blind = '../data/blind/blind_set.csv'
df_blind = pd.read_csv(path_blind).replace('[]', np.nan)
df_blind['text'] = df_blind['title'] + ' ' + df_blind['summary']

# Show info
print(f'shape: {df_blind.shape}\n\nHead: ')
display(df_blind.head())
print(f'\nInfo')
display({df_blind.info()})
print(f'Describe')
display(df_blind.describe())

shape: (3706, 10)

Head: 


,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words,text
0,1756,an enquiry concerning human understanding,david hume,NaN,NaN,the argument of the enquiry proceeds by a seri...,0,16591,2820,an enquiry concerning human understanding the ...
1,2950,anyone can whistle,arthur laurents,NaN,NaN,the story is set in an imaginary american town...,0,6990,1290,anyone can whistle the story is set in an imag...
2,4331,book of joshua,NaN,NaN,NaN,chapter 1 is the first of three important mome...,0,3339,558,book of joshua chapter 1 is the first of three...
3,4332,book of ezra,NaN,NaN,NaN,for the bible text see bible gateway opens at ...,0,3335,588,book of ezra for the bible text see bible gate...
4,4376,book of numbers,NaN,NaN,NaN,god orders moses in the wilderness of sinai to...,0,3983,709,book of numbers god orders moses in the wilder...



Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3706 entries, 0 to 3705
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 3706 non-null   int64  
 1   title              3706 non-null   object 
 2   author             2115 non-null   object 
 3   publication_date   1274 non-null   float64
 4   genres             0 non-null      object 
 5   summary            3706 non-null   object 
 6   number_of_genres   3706 non-null   int64  
 7   len_summary_char   3706 non-null   int64  
 8   len_summary_words  3706 non-null   int64  
 9   text               3706 non-null   object 
dtypes: float64(1), int64(4), object(5)
memory usage: 289.7+ KB


{None}

Describe


,id,publication_date,number_of_genres,len_summary_char,len_summary_words
count,3.706000e+03,1274.000000,3706.0,3706.000000,3706.000000
mean,1.453980e+07,1961.158556,0.0,2151.728548,375.788181
std,1.033974e+07,75.428860,0.0,2638.934366,460.046242
min,1.756000e+03,1499.000000,0.0,51.000000,11.000000
25%,4.481409e+06,1953.250000,0.0,628.000000,109.000000
50%,1.441976e+07,1986.000000,0.0,1248.000000,217.000000
75%,2.309847e+07,2004.000000,0.0,2690.250000,468.000000
max,3.713232e+07,2013.000000,0.0,46096.000000,7177.000000


# Cenário de aplicação 1:
- Deseja-se descobrir se dentre os livros sem gênero rotulado existem instâncias da categoria não ficção.
- Contudo, existem muitas amostras para um humano avaliar isso rapidamente (cerca de 3700 amostras).

## Objetivo: 
    - Encontrar um método que traga os potenciais livros que poderiam ser classificados como não ficção, reduzindo a substancialmente o trabalho humano de verificação a um volume menor.
    - Nesta etapa de triagem dos potenciais candidatos, é importante selecionar o máximo possível de potenciais instancias, mas sem deixar de lado a redução volumétrica da análise, comparada ao volume total.

## Solução proposta:
- Como é importante o alto volume de verdadeiros positivos, modelos com bom recall da classe positiva são bons candidatos para aplicação.
    - A precisão da classe positiva também é importante, para reduzir o volume de amostras em que será feito o double check humano.

- Diante das especificações, considera-se o modelo de regressão logistica com hiperparâmetros otimizados para a ótima AUPRC e threshold otimizado para maior Acurácia Balanceada adequado.
    - Esse modelo teve uma Val Acc de 95% e um recall da classe positiva e negativa também de 95%.
        - O alto recall da classe positiva atende a especificação para alto número de Verdadeiros positivos

# Resultado e estimativas
- 761 instancias (21% da base) foram separadas. 
    - Segundo as métricas desse modelo com tunning de thr, estima-se que 33% (35/(35+71), corresponde a cerca de 251 instâncias) sejam, de fato, não ficção. 
    - Segundo o recall da classe positiva, essas 251 amostras estimadas corresponderiam a 95% do volume total de instâncias de não ficção dessa base não rotulada.
        - Extrapolando as estimativas, a previsão para o número total de instancias positivas na base é de cerca de 264 (251/0,95)
            - portanto, estima-se que apenas 13 amostras positivas não foram separadas no conjunto, e estão dentre as 2915 classificadas como Ficção.
            - Segundo as estimativas a proporção de classes desse conjunto seria 264/(2915 +791) ~ 7% de amostras não ficção.
                - Essa diferença comparada aos 2,5% na base de treinamento nos aponta a duas possibilidades:
                    - Ou o modelo superestimou a classe positiva
                    - Ou a base cega não está nas mesmas proporções da base de treinamento.
                - Acredita-se que a segunda hipótese seja a verdadeira, pois o modelo de alta precisão (do próximo exercício, levanta evidencias para uma proporção maior de Não Ficções do que aquela da base de treinamento)

- Portanto, acredita-se que a tarefa tenha sido feita com exito, reduzindo cerca de 80% do trabalho do revisor, perdendo apenas 13 instâncias (5%) não separadas no conjunto.

In [3]:
# load models
base_dir = Path('../models')
loaded_models = {}
for opt_dir in base_dir.iterdir():
    opt_name = opt_dir.name
    loaded_models[opt_name] = {}
    for model_path in opt_dir.glob('*.joblib'):
        model_name = model_path.name.split('_')[0]
        model_object = joblib.load(model_path)
        loaded_models[opt_name][model_name] = model_object

# load thresholds
thr_dir = "../results/opt_auprc"
with open(os.path.join(thr_dir, "opt_thr_bal_acc.json")) as f:
    opt_thr_bal_acc = json.load(f)

# Seleção do modelo, recuperação de threshold ideal
logreg_auprc_model = loaded_models['opt_auprc']['logreg']
logreg_bal_acc_thr = opt_thr_bal_acc['logreg']['threshold']

# predição utiliando tunning
probas_logreg = logreg_auprc_model.predict_proba(df_blind['text'].astype(str).values)[:, 1]
preds_logreg = (probas_logreg >= logreg_bal_acc_thr).astype(int)
df_blind['preds_logreg_auprc_thr_opt_bal_acc'] = preds_logreg

vc_abs = df_blind['preds_logreg_auprc_thr_opt_bal_acc'].value_counts()
vc_norm = df_blind['preds_logreg_auprc_thr_opt_bal_acc'].value_counts(normalize=True).round(4)
pd.concat([vc_abs, vc_norm], axis=1, keys=['counts', 'normalized'])

,counts,normalized
preds_logreg_auprc_thr_opt_bal_acc,,
0,2915,0.7866
1,791,0.2134


# Cenário de aplicação 2 
- Deseja-se encontrar um número K de amostras para recomendação da classe Não Ficção.
    - Para esse exemplo, vamos supor K = 3.
- É importante nesse exercício que as amostras recomendadas sejam, de fato, instancias de Não Ficção. 
- Caso se tenha pouca certeza sobre a classificação, envie o interva-lo de confiança que a indicação se encontra.

# Objetivo:
- Selecionar K=3 instâncias com alta probabilidade de serem da classe não ficção. Ser preciso.
- Considerar intervalo de confiança para classificação.

# Solução proposta
- Como se deve pescar a classe positiva com precisão, não é tão relevante ter um recall baixo no modelo escolhido.
    - O modelo escolhido deve ter alta precisão da classe positiva. 
- O modelo nb_auprc teve uma precisão macro de 98% em validação cruzada. Pensando na matriz de confusão, a precisão da classe positiva também foi quase 100%, porém caiu para 6/7 quando avaliado em conjunto de teste. 
    - Ao tunar o treshold para precision macro, a nova precisão para a classe positiva foi de 8/9 ~ 89%. 
        - Tomaremos esse como o número que se deve passar como confiança, que é aquelo de probabilidade acima do threshold idealizado.
- Para a seleção dos K livros, contudo, nada impede de pegarmos aqueles com maior probabilidade calculada pelo modelo

# Resultados
- 164 exemplos foram encontrados dentro do intervalo de confiançao (proba > thr_opt -> 89% precisão)
    - É razoavel supor ao escolher as instâncias com maior probabilidade torne a recomendação ainda mais precisa.
        - Para K=3, a seguir, são apresentados top K recomendações:

In [4]:
# abrir modelo
with open(os.path.join(thr_dir, "opt_thr_precision.json")) as f:
    opt_thr_precision = json.load(f)
nb_auprc_model = loaded_models['opt_auprc']['nb']
nb_prec_thr = opt_thr_precision['nb']['threshold']

# Prediction
proba_nb_prec = nb_auprc_model.predict_proba(df_blind['text'].astype(str).values)[:, 1]
pred_nb_prec_thr = (proba_nb_prec >= nb_prec_thr).astype(int)
df_blind['proba_nb_auprc_thr_opt_precision'] = proba_nb_prec
df_blind['pred_nb_auprc_thr_opt_precision'] = pred_nb_prec_thr


df_blind['pred_nb_auprc_thr_opt_precision'].value_counts()

pred_nb_auprc_thr_opt_precision
0    3542
1     164
Name: count, dtype: int64

In [5]:
k = 3

# Show top k
df_blind.sort_values(by='proba_nb_auprc_thr_opt_precision', 
                     ascending=False)[['id', 
                                        'title', 
                                        'author', 
                                        'summary', 
                                        'proba_nb_auprc_thr_opt_precision', 
                                        'pred_nb_auprc_thr_opt_precision']
                                     ].head(k)

,id,title,author,summary,proba_nb_auprc_thr_opt_precision,pred_nb_auprc_thr_opt_precision
2188,18244814,dawkins vs gould,kim sterelny,in the introductory chapter the author points ...,0.998843,1
1460,9378842,the iq controversy the media and public policy,NaN,respondents on average identified themselves a...,0.996780,1
2854,23810295,the broken compass how british politics lost i...,peter hitchens,in chapter 1 guy fawkes gets a blackberry hitc...,0.992962,1


# Obs: para estar no intervalo de confiança estipulado, a coluna pred_nb_auprc_thr_opt_precision deve ser 1